<img src="../../../assets/images/logos/ucu_logo_clean.svg" alt="UCU Logo" width="200" style="float: right; margin: 0 0 10px 10px;"/>

### Matemáticas para Aprendizaje Automático - 2026

--------
## Laboratorio 2.1: Factorización de Matrices

#### Objetivos

- Implementar desde cero las descomposiciones LU y de Cholesky, y verificarlas contra SciPy y NumPy.
- Resolver sistemas lineales por sustitución hacia adelante y hacia atrás a partir de una factorización.
- Analizar el papel del pivoteo en la estabilidad numérica de la eliminación gaussiana.
- Calcular determinantes como producto de los pivotes de la factorización LU.

## Introducción

Factorizar una matriz consiste en escribirla como producto de matrices con
estructura conocida (triangulares, ortogonales, diagonales). La ventaja es
práctica: una vez pagado el costo de la factorización, que es $O(n^3)$, cada
nuevo sistema lineal $Ax = b$ se resuelve en $O(n^2)$ operaciones. En problemas
de aprendizaje automático esto aparece a cada paso, desde las ecuaciones
normales de mínimos cuadrados $A^\top A\,\hat{x} = A^\top b$ hasta la evaluación
de una densidad gaussiana multivariada, el muestreo de datos con una covarianza
prescrita o cada iteración de un método de Newton.

En este laboratorio se implementan la descomposición LU y la de Cholesky, y se
las usa para resolver sistemas y calcular determinantes. El hilo conductor es
que dos algoritmos matemáticamente equivalentes pueden dar respuestas muy
distintas en aritmética de punto flotante: el pivoteo, que sobre el papel es
apenas un reordenamiento de filas, decide si la eliminación gaussiana conserva
o pierde dígitos.

In [ ]:
# Librerías necesarias
import numpy as np
import scipy.linalg as la

## Parte 1: Descomposición LU

La descomposición LU escribe una matriz cuadrada $A \in \mathbb{R}^{n \times n}$
como producto de una triangular inferior $L$ con unos en la diagonal y una
triangular superior $U$:

$$A = LU, \qquad
L = \begin{bmatrix} 1 & & & \\ \ell_{21} & 1 & & \\ \vdots & & \ddots & \\ \ell_{n1} & \ell_{n2} & \cdots & 1 \end{bmatrix},
\qquad
U = \begin{bmatrix} u_{11} & u_{12} & \cdots & u_{1n} \\ & u_{22} & \cdots & u_{2n} \\ & & \ddots & \vdots \\ & & & u_{nn} \end{bmatrix}$$

La factorización es un registro de la eliminación gaussiana. En el paso $k$ se
toma $u_{kk}$ como pivote y se anulan las entradas debajo de él restando a la
fila $i > k$ la fila $k$ multiplicada por

$$\ell_{ik} = \frac{u_{ik}}{u_{kk}}$$

Los multiplicadores $\ell_{ik}$ que se van usando son exactamente las entradas
de $L$, y la matriz que queda al final del proceso es $U$. El costo es
$\tfrac{2}{3}n^3 + O(n^2)$ operaciones.

**Para qué sirve.** Con $A = LU$ disponible, el sistema $Ax = b$ se parte en dos
sistemas triangulares que cuestan $O(n^2)$ cada uno:

$$Ly = b \quad \text{(sustitución hacia adelante)}, \qquad Ux = y \quad \text{(sustitución hacia atrás)}$$

Y como $\det(L) = 1$, el determinante sale de la diagonal de $U$:

$$\det(A) = \det(L)\det(U) = \prod_{i=1}^{n} u_{ii}$$

**Cuándo falla.** El algoritmo descrito se rompe si algún pivote $u_{kk}$ es
cero, y pierde precisión si es muy chico, porque entonces
$\ell_{ik} = u_{ik}/u_{kk}$ es enorme y amplifica los errores de redondeo de
todas las filas siguientes. La respuesta estándar es el **pivoteo parcial**:
intercambiar filas para que el pivote sea el elemento de mayor valor absoluto
de la columna. Eso garantiza $|\ell_{ik}| \le 1$ y produce la factorización

$$PA = LU \qquad \Longleftrightarrow \qquad A = P^{\top} L U$$

con $P$ una matriz de permutación. Es lo que devuelve `scipy.linalg.lu`, y es
la razón por la que entrega tres matrices en lugar de dos.

En esta parte se implementa la versión sin pivoteo, se la compara contra SciPy
y se observa en qué casos falla.

### Ejercicio L2.1.1: Descomposición LU sin pivoteo

Implementá la función `lu_decomposition(A)`, que factoriza una matriz cuadrada
$A$ como $A = LU$ por eliminación gaussiana, sin pivoteo.

**a)** Recorré las columnas $k = 0, \dots, n-2$. Para cada fila $i > k$ calculá
el multiplicador $\ell_{ik} = u_{ik}/u_{kk}$, guardalo en `L[i, k]` y actualizá
la fila $i$ de $U$ restándole $\ell_{ik}$ veces la fila $k$.

**b)** La función ya trae la comprobación de pivote nulo. Mirá qué condición usa
y por qué se compara contra `np.finfo(A.dtype).eps` en lugar de contra cero.

**Nota:** trabajá sobre una copia de $A$ (`U = A.copy()`) para no modificar la
matriz que recibe la función, e inicializá `L` como la identidad.

In [ ]:
def lu_decomposition(A):
    """
    Implementa la descomposición LU sin pivoteo para una matriz cuadrada.

    Args:
        A: Matriz cuadrada numpy de tamaño n x n

    Returns:
        L: Matriz triangular inferior con unos en la diagonal
        U: Matriz triangular superior
    """
    n = A.shape[0]
    if A.shape[0] != A.shape[1]:
        raise ValueError("La matriz debe ser cuadrada")

    U = A.copy()
    L = np.eye(n)

    for k in range(n - 1):
        # Verificar si el pivote es numéricamente cero dada la precisión que se usa
        if abs(U[k, k]) < 1e3 * np.finfo(A.dtype).eps:
            raise ValueError(f"Elemento pivote U[{k},{k}] es casi cero. La matriz requiere pivoteo.")

        for i in range(k + 1, n):
            lik = ...   # COMPLETAR: multiplicador de la fila i respecto del pivote U[k, k]
            pass        # COMPLETAR: guardar lik en L[i, k] y actualizar la fila i de U

    return L, U


# ── Verificación ─────────────────────────────────────────────────────────────────
A = np.array([[2, 1, 1],
              [4, 3, 3],
              [8, 7, 9]], dtype=float)

L, U = lu_decomposition(A)
print("L =\n", L)
print("U =\n", U)
print("L @ U =\n", L @ U)

assert np.allclose(L @ U, A), "L @ U no reconstruye A"
assert np.allclose(L, np.tril(L)), "L no es triangular inferior"
assert np.allclose(np.diag(L), 1.0), "L no tiene unos en la diagonal"
assert np.allclose(U, np.triu(U)), "U no es triangular superior"
print("\nLU correcta: A = L @ U, con L triangular inferior unitaria y U triangular superior")

### Ejercicio L2.1.2: Comparación con la implementación de SciPy

SciPy trae la factorización con pivoteo parcial en `scipy.linalg.lu(A)`, que
devuelve tres matrices $P$, $L$ y $U$ tales que $A = PLU$.

**a)** Completá `compare_lu_implementations` para obtener la factorización de
SciPy, reconstruir $A$ como $PLU$ y medir el error de reconstrucción con la
norma de Frobenius $\|A - PLU\|_F$.

**b)** Compará los factores $L$ y $U$ propios contra los de SciPy. Sobre la
matriz de prueba no coinciden. Explicá por qué mirando la matriz $P$ que
devuelve SciPy.

**Nota:** la norma de Frobenius de una matriz se obtiene con
`np.linalg.norm(M, ord='fro')`. La comparación de reconstrucción debe dar cero
a menos de redondeo; la de los factores, no.

In [ ]:
def compare_lu_implementations(A):
    """
    Compara nuestra implementación de LU con la de SciPy.

    Args:
        A: Matriz a descomponer

    Returns:
        error_reconstruccion: norma de Frobenius de A - P @ L @ U (SciPy)
    """
    L_custom, U_custom = lu_decomposition(A)

    P_scipy, L_scipy, U_scipy = ...   # COMPLETAR: factorización con scipy.linalg.lu
    A_reconstruida = ...              # COMPLETAR: reconstruir A a partir de P, L y U
    error_reconstruccion = ...        # COMPLETAR: norma de Frobenius de A - A_reconstruida

    print("Matriz de permutación P de SciPy:\n", P_scipy)
    print("\nError de reconstrucción de SciPy, ||A - PLU||_F:", error_reconstruccion)
    print("Diferencia entre L propia y L de SciPy:", np.linalg.norm(L_custom - L_scipy, ord='fro'))
    print("Diferencia entre U propia y U de SciPy:", np.linalg.norm(U_custom - U_scipy, ord='fro'))
    return error_reconstruccion


# ── Verificación ─────────────────────────────────────────────────────────────────
error_reconstruccion = compare_lu_implementations(A)

assert np.isclose(error_reconstruccion, 0.0, atol=1e-12), "SciPy debería reconstruir A exactamente"
P_scipy, _, _ = la.lu(A)
assert not np.allclose(P_scipy, np.eye(A.shape[0])), \
    "En esta matriz SciPy sí permuta filas, así que P no es la identidad"
print("\nSciPy reconstruye A con error del orden de la precisión de máquina,")
print("pero sus factores difieren de los propios porque P no es la identidad.")

### Ejercicio L2.1.3: Resolución de sistemas lineales con LU

Con la factorización $A = LU$ ya calculada, resolver $Ax = b$ se reduce a dos
sistemas triangulares. La sustitución hacia adelante resuelve $Ly = b$:

$$y_i = b_i - \sum_{j<i} \ell_{ij}\, y_j$$

donde no hace falta dividir porque $\ell_{ii} = 1$. La sustitución hacia atrás
resuelve $Ux = y$ recorriendo las filas de abajo hacia arriba:

$$x_i = \frac{1}{u_{ii}}\left( y_i - \sum_{j>i} u_{ij}\, x_j \right)$$

**a)** Implementá `forward_substitution(L, b)`.

**b)** Implementá `backward_substitution(U, y)`.

**c)** Completá `solve_linear_system_lu(A, b)` encadenando ambas con
`lu_decomposition`.

**Nota:** el resultado se compara contra `np.linalg.solve`, que usa LU con
pivoteo internamente.

In [ ]:
def forward_substitution(L, b):
    """
    Resuelve el sistema Ly = b mediante sustitución hacia adelante.
    L debe ser triangular inferior con unos en la diagonal.

    Args:
        L: Matriz triangular inferior con unos en la diagonal
        b: Vector del lado derecho

    Returns:
        y: Solución del sistema Ly = b
    """
    n = L.shape[0]
    y = np.zeros_like(b, dtype=float)

    pass  # COMPLETAR: sustitución hacia adelante
    return y


def backward_substitution(U, y):
    """
    Resuelve el sistema Ux = y mediante sustitución hacia atrás.
    U debe ser triangular superior.

    Args:
        U: Matriz triangular superior
        y: Vector del lado derecho

    Returns:
        x: Solución del sistema Ux = y
    """
    n = U.shape[0]
    x = np.zeros_like(y, dtype=float)

    pass  # COMPLETAR: sustitución hacia atrás (recordar dividir por U[i, i])
    return x


def solve_linear_system_lu(A, b):
    """
    Resuelve el sistema Ax = b utilizando la descomposición LU.

    Args:
        A: Matriz de coeficientes
        b: Vector del lado derecho

    Returns:
        x: Solución del sistema
    """
    L, U = ...   # COMPLETAR: factorizar A
    y = ...      # COMPLETAR: resolver Ly = b
    x = ...      # COMPLETAR: resolver Ux = y
    return x


# ── Verificación ─────────────────────────────────────────────────────────────────
A = np.array([[2, 1, 1],
              [4, 3, 3],
              [8, 7, 9]], dtype=float)
b = np.array([5, 16, 41], dtype=float)

x_custom = solve_linear_system_lu(A, b)
x_numpy = np.linalg.solve(A, b)
print("Solución propia      :", x_custom)
print("np.linalg.solve      :", x_numpy)
print("Residuo ||A x - b||_2:", np.linalg.norm(A @ x_custom - b))

assert np.allclose(x_custom, x_numpy), "La solución propia no coincide con np.linalg.solve"
assert np.allclose(A @ x_custom, b), "A x no reproduce b"

### Ejercicio L2.1.4: Un caso donde hace falta pivotear

Considerá la matriz

$$A = \begin{bmatrix} 0 & 1 \\ 1 & 1 \end{bmatrix}$$

Es invertible ($\det A = -1$), pero su entrada $a_{11}$ es cero, así que la
eliminación gaussiana sin pivoteo se detiene en el primer paso.

**a)** Completá `test_lu_stability` para intentar la factorización propia
capturando el `ValueError`, y para factorizar la misma matriz con SciPy.

**b)** Verificá que la reconstrucción $PLU$ de SciPy sí recupera $A$.

**Nota:** la función devuelve `True` si la implementación propia falló, para
que el bloque de verificación pueda comprobarlo.

In [ ]:
def test_lu_stability():
    """
    Prueba un caso donde la descomposición LU sin pivoteo presenta problemas.

    Returns:
        fallo_propio: True si lu_decomposition levantó ValueError
        error_scipy: norma de Frobenius de A - PLU con SciPy
    """
    A_problematic = np.array([[0, 1],
                              [1, 1]], dtype=float)
    print("Matriz problemática:\n", A_problematic)

    fallo_propio = False
    try:
        L, U = lu_decomposition(A_problematic)
        print("L @ U =\n", L @ U)
    except ValueError as e:
        fallo_propio = True
        print("Error con la implementación propia:", e)

    P_scipy, L_scipy, U_scipy = ...   # COMPLETAR: factorización de SciPy
    A_reconstruida = ...              # COMPLETAR: reconstruir A
    error_scipy = ...                 # COMPLETAR: norma de Frobenius del error

    print("\nDescomposición LU de SciPy, que incluye pivoteo:")
    print("P =\n", P_scipy)
    print("Reconstrucción P @ L @ U =\n", A_reconstruida)
    return fallo_propio, error_scipy


# ── Verificación ─────────────────────────────────────────────────────────────────
fallo_propio, error_scipy = test_lu_stability()

assert fallo_propio, "La implementación sin pivoteo debería fallar sobre esta matriz"
assert np.isclose(error_scipy, 0.0, atol=1e-12), "SciPy debería reconstruir A con pivoteo"
print("\nSin pivoteo el algoritmo se detiene; con pivoteo la matriz se factoriza sin problema.")

### Ejercicio L2.1.5: Determinante a partir de la factorización LU

Como $\det(L) = 1$, el determinante de $A$ es el producto de los pivotes:

$$\det(A) = \prod_{i=1}^{n} u_{ii}$$

Implementá `determinant_custom(A)` usando `lu_decomposition` y compará el
resultado contra `np.linalg.det`.

**Nota:** los dos valores no van a coincidir bit a bit, porque `np.linalg.det`
usa pivoteo y por lo tanto ejecuta otras operaciones en otro orden. La
comparación se hace con `np.isclose`, no con `==`.

In [ ]:
def determinant_custom(A):
    """
    Calcula det(A) como el producto de los pivotes de la factorización LU.

    Args:
        A: Matriz cuadrada

    Returns:
        determinant: valor del determinante
    """
    L, U = ...              # COMPLETAR: factorizar A
    determinant = ...       # COMPLETAR: producto de los elementos de la diagonal de U
    return determinant


# ── Verificación ─────────────────────────────────────────────────────────────────
det_custom = determinant_custom(A)
det_numpy = np.linalg.det(A)
print("Determinante por LU propia :", det_custom)
print("np.linalg.det              :", det_numpy)
print("¿Coinciden bit a bit?      :", det_custom == det_numpy)

assert np.isclose(det_custom, det_numpy), "El determinante no coincide con np.linalg.det"

## Parte 2: Descomposición de Cholesky

Cuando $A$ es **simétrica y definida positiva** ($A = A^\top$ y $x^\top A x > 0$
para todo $x \neq 0$), existe una única matriz triangular inferior $L$ con
diagonal positiva tal que

$$A = L L^{\top}$$

Es el caso particular de LU que aprovecha la simetría: en lugar de guardar $L$ y
$U$ por separado, alcanza con un solo factor. El costo baja a
$\tfrac{1}{3}n^3$ operaciones, la mitad que LU, y no hace falta pivotear, porque
la definición positiva garantiza que los pivotes nunca se anulan.

Las entradas de $L$ se obtienen recorriendo la matriz por filas:

$$\ell_{jj} = \sqrt{a_{jj} - \sum_{k<j} \ell_{jk}^2},
\qquad
\ell_{ij} = \frac{1}{\ell_{jj}}\left( a_{ij} - \sum_{k<j} \ell_{ik}\ell_{jk} \right) \quad (i > j)$$

La raíz cuadrada de la primera fórmula da un criterio numérico útil: si en algún
paso el radicando es negativo, la matriz no era definida positiva. Por eso
`np.linalg.cholesky` se usa en la práctica como test de definición positiva.

**Dónde aparece en aprendizaje automático.** Toda matriz de covarianza es
simétrica y semidefinida positiva. Cholesky permite evaluar la densidad
gaussiana multivariada sin invertir $\Sigma$, y permite generar muestras con una
covarianza prescrita: si $z \sim \mathcal{N}(0, I)$ y $\Sigma = LL^\top$,
entonces $x = Lz$ cumple

$$\operatorname{Cov}(x) = L\,\operatorname{Cov}(z)\,L^\top = L I L^\top = \Sigma$$

### Ejercicio L2.1.6: Implementación de la descomposición de Cholesky

**a)** Implementá `is_symmetric(A, tol)`, que compara $A$ con $A^\top$ a menos
de una tolerancia.

**b)** Implementá `is_positive_definite(A)` comprobando el signo de los valores
propios.

**c)** Implementá `cholesky_decomposition(A)` con las fórmulas de la
introducción. Las verificaciones de matriz cuadrada, simétrica y definida
positiva ya están escritas.

**Nota:** una matriz simétrica es definida positiva si y solo si todos sus
valores propios son positivos. Podés obtenerlos con `la.eigvals(A)`.

In [ ]:
def is_symmetric(A, tol=1e-8):
    """
    Verifica si una matriz es simétrica.

    Args:
        A: Matriz a verificar
        tol: Tolerancia absoluta para la comparación

    Returns:
        bool: True si la matriz es simétrica
    """
    result = ...   # COMPLETAR: comparar A con su transpuesta
    return result


def is_positive_definite(A):
    """
    Verifica si una matriz es definida positiva comprobando sus valores propios.

    Args:
        A: Matriz a verificar

    Returns:
        bool: True si la matriz es definida positiva
    """
    # SUGERENCIA: una matriz simétrica es definida positiva si todos sus valores propios son positivos
    result = ...   # COMPLETAR
    return result


def cholesky_decomposition(A):
    """
    Implementa la descomposición de Cholesky para una matriz simétrica definida positiva.

    Args:
        A: Matriz simétrica definida positiva de tamaño n x n

    Returns:
        L: Matriz triangular inferior tal que A = L @ L.T
    """
    n = A.shape[0]

    if A.shape[0] != A.shape[1]:
        raise ValueError("La matriz debe ser cuadrada")
    if not is_symmetric(A):
        raise ValueError("La matriz debe ser simétrica")
    if not is_positive_definite(A):
        raise ValueError("La matriz debe ser definida positiva")

    L = np.zeros_like(A, dtype=float)

    for i in range(n):
        for j in range(i + 1):
            pass  # COMPLETAR: calcular L[i, j] distinguiendo el caso diagonal (i == j)
    return L


# ── Verificación ─────────────────────────────────────────────────────────────────
A_spd = np.array([[4, 12, -16],
                  [12, 37, -43],
                  [-16, -43, 98]], dtype=float)

L_chol = cholesky_decomposition(A_spd)
print("L =\n", L_chol)
print("L @ L.T =\n", L_chol @ L_chol.T)

assert is_symmetric(A_spd) and is_positive_definite(A_spd), "La matriz de prueba debe ser SDP"
assert np.allclose(L_chol @ L_chol.T, A_spd), "L @ L.T no reconstruye A"
assert np.allclose(L_chol, np.tril(L_chol)), "L no es triangular inferior"
assert np.all(np.diag(L_chol) > 0), "La diagonal de L debe ser positiva"

### Ejercicio L2.1.7: Comparación con la implementación de NumPy

`np.linalg.cholesky(A)` devuelve el mismo factor triangular inferior $L$ con
diagonal positiva. A diferencia de LU, acá la factorización es única, así que
los dos resultados deben coincidir a menos de redondeo.

**a)** Completá `compare_cholesky_implementations` para obtener el factor de
NumPy y medir $\|L_{\text{propia}} - L_{\text{NumPy}}\|_F$.

**b)** Calculá el error de reconstrucción $\|A - LL^\top\|_F$ de cada
implementación.

**Nota:** revisá la documentación de `np.linalg.cholesky` para confirmar si
devuelve el factor inferior o el superior.

In [ ]:
def compare_cholesky_implementations(A):
    """
    Compara nuestra implementación de Cholesky con la de NumPy.

    Args:
        A: Matriz simétrica definida positiva

    Returns:
        diferencia_factores: ||L_custom - L_numpy||_F
        error_propio: ||A - L_custom @ L_custom.T||_F
        error_numpy: ||A - L_numpy @ L_numpy.T||_F
    """
    L_custom = cholesky_decomposition(A)

    L_numpy = ...              # COMPLETAR: factor de Cholesky con NumPy
    diferencia_factores = ...  # COMPLETAR: norma de Frobenius de L_custom - L_numpy
    error_propio = ...         # COMPLETAR: ||A - L_custom @ L_custom.T||_F
    error_numpy = ...          # COMPLETAR: ||A - L_numpy @ L_numpy.T||_F

    print("Diferencia entre factores ||L_custom - L_numpy||_F:", diferencia_factores)
    print("Error de reconstrucción propio :", error_propio)
    print("Error de reconstrucción NumPy  :", error_numpy)
    return diferencia_factores, error_propio, error_numpy


# ── Verificación ─────────────────────────────────────────────────────────────────
diferencia_factores, error_propio, error_numpy = compare_cholesky_implementations(A_spd)

assert np.isclose(diferencia_factores, 0.0, atol=1e-10), \
    "Cholesky es única: los factores deberían coincidir"
assert error_propio < 1e-10 and error_numpy < 1e-10, "Alguna reconstrucción no recupera A"

### Ejercicio L2.1.8: Resolución de sistemas lineales con Cholesky

Con $A = LL^\top$, el sistema $Ax = b$ se resuelve igual que con LU, pero los
dos factores triangulares son $L$ y $L^\top$:

$$Ly = b, \qquad L^{\top} x = y$$

La diferencia con el Ejercicio L2.1.3 es que acá $L$ **no** tiene unos en la
diagonal, así que la sustitución hacia adelante también divide por $\ell_{ii}$.

Implementá `solve_linear_system_cholesky(A, b)` reutilizando tu
`cholesky_decomposition` y las sustituciones que ya escribiste, o escribiendo
los bucles de nuevo.

**Nota:** `backward_substitution` del Ejercicio L2.1.3 sirve tal cual para
$L^\top x = y$, porque $L^\top$ es triangular superior. Para $Ly = b$ hace falta
una sustitución hacia adelante que divida por la diagonal.

In [ ]:
def solve_linear_system_cholesky(A, b):
    """
    Resuelve el sistema Ax = b utilizando la descomposición de Cholesky.

    Args:
        A: Matriz de coeficientes (simétrica definida positiva)
        b: Vector del lado derecho

    Returns:
        x: Solución del sistema
    """
    L = ...   # COMPLETAR: factor de Cholesky de A
    n = L.shape[0]

    # Sustitución hacia adelante: L y = b  (la diagonal de L no es unitaria)
    y = np.zeros_like(b, dtype=float)
    pass      # COMPLETAR

    # Sustitución hacia atrás: L.T x = y
    x = ...   # COMPLETAR
    return x


# ── Verificación ─────────────────────────────────────────────────────────────────
A_sys = np.array([[4, 1, 1],
                  [1, 3, 1],
                  [1, 1, 5]], dtype=float)
b_sys = np.array([6, 5, 7], dtype=float)

x_chol = solve_linear_system_cholesky(A_sys, b_sys)
x_ref = np.linalg.solve(A_sys, b_sys)
print("Solución por Cholesky :", x_chol)
print("np.linalg.solve       :", x_ref)
print("Residuo ||A x - b||_2 :", np.linalg.norm(A_sys @ x_chol - b_sys))

assert np.allclose(x_chol, x_ref), "La solución por Cholesky no coincide con np.linalg.solve"
assert np.allclose(A_sys @ x_chol, b_sys), "A x no reproduce b"

## Conclusiones

En este laboratorio hemos explorado:

1. **Descomposición LU**: se implementó la eliminación gaussiana registrando los
   multiplicadores en $L$, y se la usó para resolver sistemas en $O(n^2)$ por
   cada lado derecho y para calcular determinantes como producto de pivotes.
2. **Pivoteo**: un pivote nulo detiene el algoritmo y uno chico degrada la
   precisión, porque el multiplicador $\ell_{ik} = u_{ik}/u_{kk}$ se vuelve
   enorme. El pivoteo parcial lo acota por 1 y produce la factorización
   $PA = LU$ que devuelve SciPy.
3. **Descomposición de Cholesky**: para matrices simétricas definidas positivas
   alcanza un único factor, $A = LL^\top$, con la mitad del costo de LU. La
   factorización es única, así que la implementación propia coincide con la de
   NumPy, y la raíz cuadrada de la recurrencia da además un criterio práctico
   para detectar que una matriz no es definida positiva.
4. **Verificación contra librerías**: en cada ejercicio la implementación manual
   se contrastó con SciPy o NumPy. Cuando los resultados difieren, como en LU
   con y sin pivoteo, la diferencia tiene una explicación concreta y no es un
   error de implementación.

## Declaración de uso de inteligencia artificial

> **Política del curso (syllabus).** Se permite usar herramientas de IA (ChatGPT, Copilot, Claude, etc.)
> *como apoyo para el aprendizaje*: entender conceptos, explorar ideas, depurar código o buscar
> explicaciones alternativas. **No** se permite usarlas para **resolver los ejercicios evaluados** ni
> para **verificar las respuestas antes de entregar**. Se espera que cada estudiante resuelva todos los
> problemas por sí mismo/a.

Completá esta declaración **escribiendo tu respuesta** donde aparece «…» (doble clic en esta celda para
editarla y luego Ctrl/Cmd + Enter para volver a renderizarla):

**1. ¿Usaste herramientas de IA en este laboratorio?** (Sí / No): «…»

**2. ¿Cuál(es)?** (ChatGPT, Copilot, Claude, …; escribí "ninguna" si no usaste): «…»

**3. ¿Para qué la(s) usaste?** Usos permitidos: entender conceptos, explorar ideas, depurar código,
buscar explicaciones alternativas. Escribí los que apliquen: «…»

**4. Detalle breve:** «En qué ejercicios y de qué manera. Ej.: "Usé Claude para entender la definición de
matriz definida positiva antes del ejercicio L2.1.6."»

**Declaración de honestidad académica.** Declaro que resolví los ejercicios de este laboratorio por mí
mismo/a y que no utilicé herramientas de IA para resolver los ejercicios evaluados ni para verificar mis
respuestas antes de entregar, de acuerdo con la política del curso.

**Nombre y apellido:** «…»          **Fecha:** «…»